# Mediterranean Cuisine RAG Pipeline
## COMP 647-02: Transforming Text into Meaning

This notebook ties together the complete RAG (Retrieval-Augmented Generation) pipeline for a Mediterranean cuisine question-answering system.

### Pipeline Components
1. **Corpus Collection** — 105 documents from Wikipedia, Wikibooks, and blog sources
2. **Chunking** — 4 strategies (section-based, fixed-200, fixed-500, sentence-based)
3. **Embedding** — 3 models (MiniLM, MPNet, BGE) with FAISS indexing
4. **Retrieval & Ranking** — Vector, BM25, and Hybrid (RRF) retrieval returning top-5 chunks
5. **Prompting & Generation** — Qwen/Qwen2.5-0.5B-Instruct with 3 prompt strategies
6. **Evaluation** — Precision@5, Recall@5, MRR, ROUGE-L, BERTScore, Faithfulness

### Best Configuration (from evaluation)
- **Retrieval:** Vector + all-mpnet-base-v2 + section_based (MRR 1.0, Recall@5 0.98)
- **Generation:** Structured prompting (ROUGE-L 0.25)

In [ ]:
# Imports and setup
import os
import json
import time
import numpy as np

# Set working directory to the code folder
CODE_DIR = os.path.dirname(os.path.abspath("__file__"))
os.chdir(CODE_DIR)

print(f"Working directory: {os.getcwd()}")
print(f"Files: {sorted([f for f in os.listdir('.') if f.endswith(('.py', '.json', '.csv'))])}")

---
## 1. Background Corpus (Deliverable 1)

Built using `build_corpus.py`. Scraped 80 documents from 3 publicly available sources:
- **Wikipedia** — cuisine overview articles, dish pages, ingredient pages
- **Wikibooks (Cookbook)** — recipe and technique pages
- **Around the World in 80 Cuisines Blog** — regional cuisine articles

Each document is saved as a `.txt` file in `corpus/` with a structured metadata header (TITLE, SOURCE, URL, SCRAPED, LICENCE).

In [ ]:
# 1.1 — Corpus overview
import csv

corpus_files = sorted([f for f in os.listdir("corpus") if f.endswith(".txt") and f != "corpus_combined.txt"])
print(f"Total corpus documents: {len(corpus_files)}")

# Count by source
if os.path.exists("corpus_manifest.csv"):
    with open("corpus_manifest.csv", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    sources = {}
    for r in rows:
        src = r.get("source", "unknown")
        sources[src] = sources.get(src, 0) + 1
    print(f"\nDocuments per source:")
    for src, count in sorted(sources.items()):
        print(f"  {src:12s}: {count}")

# Show a sample file header
sample = corpus_files[0]
with open(os.path.join("corpus", sample), encoding="utf-8") as f:
    lines = f.readlines()[:8]
print(f"\nSample file header ({sample}):")
for line in lines:
    print(f"  {line.rstrip()}")

---
## 2. Chunking (Component 1)

Built using `chunker.py`. Splits corpus documents into retrieval-ready chunks.

**Strategies implemented:**
| Strategy | Description | Output File |
|---|---|---|
| **Section-based** (primary) | Respects document headings, adaptive 100-500 word sizing, context headers | `chunks.json` |
| **Fixed-size (200w)** | Fixed 200-word chunks with 50-word overlap | `chunks_fixed_200.json` |
| **Fixed-size (500w)** | Fixed 500-word chunks with 50-word overlap | `chunks_fixed_500.json` |
| **Sentence-based** | Groups sentences to ~300 words at sentence boundaries | `chunks_sentence.json` |

Each chunk includes a context header: `[Source: {source} | Title: {title} | Section: {section}]`

In [ ]:
# 2.1 — Load and inspect chunks across all strategies
CHUNK_FILES = {
    "section_based":  "chunks.json",
    "fixed_200":      "chunks_fixed_200.json",
    "fixed_500":      "chunks_fixed_500.json",
    "sentence_based": "chunks_sentence.json",
}

print(f"{'Strategy':<16s} {'Chunks':>7s} {'Avg Words':>10s} {'Min':>5s} {'Max':>5s}")
print("-" * 50)

for strategy, filename in CHUNK_FILES.items():
    if os.path.exists(filename):
        with open(filename, encoding="utf-8") as f:
            chunks = json.load(f)
        word_counts = [c["word_count"] for c in chunks]
        print(f"{strategy:<16s} {len(chunks):>7d} {np.mean(word_counts):>10.1f} {min(word_counts):>5d} {max(word_counts):>5d}")
    else:
        print(f"{strategy:<16s}   (not yet generated — run: python chunker.py --strategy {strategy})")

In [ ]:
# 2.2 — Show a sample chunk (section-based)
with open("chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

sample_chunk = chunks[0]
print("Sample chunk:")
print(json.dumps(sample_chunk, indent=2, ensure_ascii=False)[:800])

---
## 3. Vectorisation / Embedding (Component 2)

Built using `embedder.py`. Converts chunks into dense vector representations using sentence-transformers, then builds FAISS indices for fast similarity search.

**Embedding models tested:**
| Model | Dimensions | Description |
|---|---|---|
| `all-MiniLM-L6-v2` | 384 | Lightweight baseline |
| `all-mpnet-base-v2` | 768 | Best quality (selected) |
| `BAAI/bge-small-en-v1.5` | 384 | Retrieval specialist |

FAISS indices are stored in `indices/` as `faiss_{model}_{strategy}.bin` with corresponding `mapping_{model}_{strategy}.json` files.

In [ ]:
# 3.1 — Verify FAISS indices exist
INDEX_DIR = "indices"

if os.path.exists(INDEX_DIR):
    index_files = sorted(os.listdir(INDEX_DIR))
    faiss_files = [f for f in index_files if f.startswith("faiss_")]
    mapping_files = [f for f in index_files if f.startswith("mapping_")]
    
    print(f"FAISS indices: {len(faiss_files)}")
    for f in faiss_files:
        size_mb = os.path.getsize(os.path.join(INDEX_DIR, f)) / (1024 * 1024)
        print(f"  {f} ({size_mb:.1f} MB)")
    
    print(f"\nID mappings: {len(mapping_files)}")
    for f in mapping_files:
        with open(os.path.join(INDEX_DIR, f), encoding="utf-8") as fh:
            mapping = json.load(fh)
        print(f"  {f} ({len(mapping)} vectors)")
else:
    print("Indices not found. Run: python embedder.py --run-all")

---
## 4. Retrieval & Ranking (Component 3)

Built using `retriever.py`. Three retrieval methods, each returning **at most 5 chunks**:

| Method | Description |
|---|---|
| **Vector** | Cosine similarity via FAISS on sentence-transformer embeddings |
| **BM25** | Keyword-based ranking using BM25Okapi (TF-IDF family) |
| **Hybrid (RRF)** | Reciprocal Rank Fusion combining vector + BM25 scores |

In [ ]:
# 4.1 — Import retriever functions and set up best retrieval config
from retriever import (
    load_chunks, setup_vector, setup_bm25,
    retrieve_vector, retrieve_bm25, retrieve_hybrid,
    MODELS, CHUNK_FILES as RETRIEVER_CHUNK_FILES
)

# Best config from evaluation
BEST_RETRIEVAL_METHOD = "vector"
BEST_EMBEDDING_MODEL = "mpnet"
BEST_CHUNKING = "section_based"
K = 5  # at most 5 chunks

print("Setting up retrieval with best config:")
print(f"  Method:    {BEST_RETRIEVAL_METHOD}")
print(f"  Model:     {MODELS[BEST_EMBEDDING_MODEL]}")
print(f"  Chunking:  {BEST_CHUNKING}")
print(f"  Top-K:     {K}")

# Load chunks for text lookup
chunk_file = RETRIEVER_CHUNK_FILES[BEST_CHUNKING]
chunks_data = load_chunks(chunk_file)
chunks_lookup = {c["chunk_id"]: c for c in chunks_data}
print(f"\nLoaded {len(chunks_data)} chunks from {chunk_file}")

# Setup vector index
vec_setup = setup_vector(BEST_EMBEDDING_MODEL, BEST_CHUNKING)

In [ ]:
# 4.2 — Demo: retrieve top-5 chunks for a sample query
demo_query = "What are the main ingredients of hummus and where did it originate?"
print(f"Query: {demo_query}\n")

start = time.time()
retrieved = retrieve_vector(demo_query, vec_setup, k=K)
retrieval_time = time.time() - start

print(f"Retrieved {len(retrieved)} chunks in {retrieval_time:.2f}s:\n")
for hit in retrieved:
    chunk = chunks_lookup.get(hit["chunk_id"], {})
    title = chunk.get("doc_title", "?")
    section = chunk.get("section", "?")
    print(f"  Rank {hit['rank']}: [{hit['score']:.4f}] {hit['chunk_id']}")
    print(f"         {title} / {section} ({chunk.get('word_count', 0)} words)")
    print(f"         {chunk.get('text', '')[:120]}...")
    print()

---
## 5. Prompting & Generation (Component 4)

Built using `generator.py`. Uses **Qwen/Qwen2.5-0.5B-Instruct** (as required by the coursework) for CPU inference.

**Three prompt strategies:**
| Strategy | Description |
|---|---|
| **Zero-shot** | Context + question, no examples |
| **Few-shot** | Includes one worked example before the real question |
| **Structured** | Asks the model to extract facts first, then write a concise answer |

In [ ]:
# 5.1 — Load LLM (Qwen2.5-0.5B-Instruct)
from generator import (
    load_model, format_context, build_messages, generate_answer,
    MODEL_NAME, SYSTEM_PROMPTS, GENERATION_CONFIG
)

print(f"LLM: {MODEL_NAME}")
print(f"Generation config: {GENERATION_CONFIG}")
print(f"\nPrompt strategies available: {list(SYSTEM_PROMPTS.keys())}")

llm_model, tokenizer = load_model()

In [ ]:
# 5.2 — Demo: generate an answer using the retrieved chunks
BEST_PROMPT_STRATEGY = "structured"

# Prepare context from retrieved chunks
retrieved_chunks = []
for hit in retrieved:
    cid = hit["chunk_id"]
    chunk = chunks_lookup.get(cid, {})
    retrieved_chunks.append({
        "doc_id": cid,
        "text": chunk.get("text", ""),
        "source": chunk.get("source", ""),
        "title": chunk.get("doc_title", ""),
    })

context_str = format_context(retrieved_chunks)
messages = build_messages(BEST_PROMPT_STRATEGY, context_str, demo_query)

print(f"Strategy: {BEST_PROMPT_STRATEGY}")
print(f"Generating answer for: {demo_query}\n")

answer, gen_time = generate_answer(llm_model, tokenizer, messages)

print(f"Answer ({gen_time:.1f}s):")
print(f"  {answer}")

---
## 6. End-to-End Inference Pipeline (Deliverable 5)

This is the core function for the live demonstration. Given an **unlabelled test set** (queries only, no gold answers), it runs the full RAG pipeline and produces output in the **required format**.

**Input format** (provided by marking team):
```json
{"queries": [{"query_id": "0", "query": "..."}]}
```

**Output format** (Deliverable 5):
```json
{"results": [{"query_id": "0", "query": "...", "response": "...", "retrieved_context": [{"doc_id": "000", "text": "..."}]}]}
```

The `doc_id` in the output uses sequential indices (`"000"`, `"001"`, ...) per the required format specification.

In [ ]:
# 6.1 — End-to-end inference function
def run_inference(input_file, output_file,
                  retrieval_method="vector",
                  embedding_model="mpnet",
                  chunking_strategy="section_based",
                  prompt_strategy="structured",
                  k=5):
    """
    Run the full RAG inference pipeline on an unlabelled test set.
    
    Reads queries from input_file, runs retrieval + generation,
    and writes results to output_file in the required Deliverable 5 format.
    
    Parameters
    ----------
    input_file : str
        Path to input JSON: {"queries": [{"query_id": "...", "query": "..."}]}
    output_file : str
        Path to write output JSON (Deliverable 5 format)
    retrieval_method : str
        One of "vector", "bm25", "hybrid"
    embedding_model : str
        One of "mpnet", "minilm", "bge"
    chunking_strategy : str
        One of "section_based", "fixed_200", "fixed_500", "sentence_based"
    prompt_strategy : str
        One of "zero_shot", "few_shot", "structured"
    k : int
        Number of chunks to retrieve (at most 5 per coursework requirement)
    """
    # --- Load input queries ---
    with open(input_file, encoding="utf-8") as f:
        data = json.load(f)
    queries = data["queries"]
    print(f"Loaded {len(queries)} queries from {input_file}")
    
    # --- Load chunks ---
    chunk_file = RETRIEVER_CHUNK_FILES[chunking_strategy]
    chunks = load_chunks(chunk_file)
    cl = {c["chunk_id"]: c for c in chunks}
    
    # --- Setup retrieval ---
    vs = None
    bs = None
    if retrieval_method in ("vector", "hybrid"):
        vs = setup_vector(embedding_model, chunking_strategy)
    if retrieval_method in ("bm25", "hybrid"):
        bs = setup_bm25(chunking_strategy)
    
    # --- Setup generation (reuse if already loaded) ---
    try:
        model = llm_model
        tok = tokenizer
        print(f"Reusing loaded LLM: {MODEL_NAME}")
    except NameError:
        print(f"Loading LLM: {MODEL_NAME}")
        model, tok = load_model()
    
    # --- Process each query ---
    results = []
    total_start = time.time()
    
    for i, q in enumerate(queries):
        query_id = q["query_id"]
        query_text = q["query"]
        
        # Retrieve
        t0 = time.time()
        if retrieval_method == "vector":
            hits = retrieve_vector(query_text, vs, k=k)
        elif retrieval_method == "bm25":
            hits = retrieve_bm25(query_text, bs, k=k)
        elif retrieval_method == "hybrid":
            hits = retrieve_hybrid(query_text, vs, bs, k=k)
        retrieval_time = time.time() - t0
        
        # Resolve chunk texts
        ret_chunks = []
        for hit in hits:
            cid = hit["chunk_id"]
            chunk = cl.get(cid, {})
            ret_chunks.append({
                "doc_id": cid,
                "text": chunk.get("text", ""),
            })
        
        # Generate
        context_str = format_context(ret_chunks)
        msgs = build_messages(prompt_strategy, context_str, query_text)
        response, gen_time = generate_answer(model, tok, msgs)
        
        # Format output with sequential doc_ids (required format)
        formatted_context = []
        for idx, rc in enumerate(ret_chunks):
            formatted_context.append({
                "doc_id": f"{idx:03d}",
                "text": rc["text"],
            })
        
        results.append({
            "query_id": query_id,
            "query": query_text,
            "response": response,
            "retrieved_context": formatted_context,
        })
        
        print(f"  [{i+1}/{len(queries)}] Q{query_id} "
              f"(retrieval: {retrieval_time:.2f}s, generation: {gen_time:.1f}s)")
    
    total_time = time.time() - total_start
    
    # --- Write output ---
    output = {"results": results}
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    
    print(f"\nInference complete!")
    print(f"  Queries processed: {len(results)}")
    print(f"  Total time: {total_time:.1f}s")
    print(f"  Output saved: {output_file}")
    
    return output

print("run_inference() function defined.")

### 6.2 — Run on Benchmark Queries

Run the inference pipeline on our 15 benchmark queries to verify the full pipeline works end-to-end.

In [ ]:
# 6.2 — Run inference on benchmark queries
benchmark_output = run_inference(
    input_file="rag_benchmark_queries.json",
    output_file="benchmark_output.json",
    retrieval_method="vector",
    embedding_model="mpnet",
    chunking_strategy="section_based",
    prompt_strategy="structured",
    k=5,
)

In [ ]:
# 6.3 — Inspect a sample result from the benchmark output
sample = benchmark_output["results"][0]
print("Sample output (query 0):")
print(json.dumps(sample, indent=2, ensure_ascii=False)[:1000])

### 6.4 — Run on Unseen Test Set (Live Demo)

**This is the cell to use during the live demonstration.**

When the marking team provides a test set JSON file, place it in the `code/` directory and update the `INPUT_FILE` path below. The output will be written in the exact required format.

In [ ]:
# ============================================================
# LIVE DEMO — UPDATE THIS PATH WITH THE TEST SET FILE
# ============================================================
INPUT_FILE = "test_queries.json"    # <-- replace with actual test file path
OUTPUT_FILE = "test_output.json"    # <-- output file for submission
# ============================================================

# Run end-to-end inference on the test set
test_output = run_inference(
    input_file=INPUT_FILE,
    output_file=OUTPUT_FILE,
    retrieval_method="vector",       # best retrieval method
    embedding_model="mpnet",         # best embedding model
    chunking_strategy="section_based",  # best chunking strategy
    prompt_strategy="structured",    # best prompt strategy
    k=5,                             # at most 5 chunks
)

---
## 7. Evaluation (Component 5)

Built using `evaluator.py`. Evaluates both retrieval quality and generation quality.

**Retrieval metrics:**
| Metric | Description |
|---|---|
| **Precision@5** | Fraction of retrieved chunks that match gold-standard |
| **Recall@5** | Fraction of gold-standard chunks found in retrieved set |
| **MRR** | Mean Reciprocal Rank — how high the first relevant chunk ranks |

**Generation metrics:**
| Metric | Description |
|---|---|
| **ROUGE-L** | Longest common subsequence overlap with gold answer |
| **BERTScore F1** | Semantic similarity via distilbert-base-uncased |
| **Faithfulness** | Fraction of answer content words grounded in retrieved context |

In [ ]:
# 7.1 — Run evaluation on benchmark results (using pre-computed results)
from evaluator import (
    load_gold_standard, evaluate_retrieval, load_chunks as eval_load_chunks,
    compute_rouge_l, compute_bert_score, compute_faithfulness,
    chunks_match, get_content_words
)

# Load gold standard
gold_standard = load_gold_standard()
print(f"Loaded {len(gold_standard)} gold-standard queries\n")

# --- Retrieval evaluation on our benchmark output ---
print("=" * 60)
print("  RETRIEVAL EVALUATION (benchmark output)")
print("=" * 60)

# Build retrieval results in the format evaluate_retrieval expects
eval_chunks_lookup = eval_load_chunks("section_based")
retrieval_for_eval = []
for result in benchmark_output["results"]:
    # Map sequential doc_ids back to chunk_ids using the chunks we retrieved
    # We need to re-retrieve to get chunk_ids (or we store them)
    query_text = result["query"]
    hits = retrieve_vector(query_text, vec_setup, k=5)
    retrieval_for_eval.append({
        "query_id": result["query_id"],
        "query": result["query"],
        "retrieved": hits,
    })

ret_metrics = evaluate_retrieval(retrieval_for_eval, gold_standard, eval_chunks_lookup)

print(f"  Precision@5: {ret_metrics['avg_precision_at_5']:.4f}")
print(f"  Recall@5:    {ret_metrics['avg_recall_at_5']:.4f}")
print(f"  MRR:         {ret_metrics['mrr']:.4f}")

In [ ]:
# 7.2 — Generation evaluation (ROUGE-L, BERTScore, Faithfulness)
print("=" * 60)
print("  GENERATION EVALUATION (benchmark output)")
print("=" * 60)

# Prepare gold and generated answers (aligned by query_id)
gold_answers = [g["response"] for g in gold_standard]
gen_answers = [r["response"] for r in benchmark_output["results"]]
gen_contexts = [r["retrieved_context"] for r in benchmark_output["results"]]

# ROUGE-L
print("\n  Computing ROUGE-L...")
rouge_scores, rouge_avg = compute_rouge_l(gold_answers, gen_answers)
print(f"  ROUGE-L F1: {rouge_avg:.4f}")

# BERTScore
print("\n  Computing BERTScore...")
bert_scores, bert_avg = compute_bert_score(gold_answers, gen_answers)
print(f"  BERTScore F1: {bert_avg:.4f}")

# Faithfulness
print("\n  Computing Faithfulness...")
faith_scores, faith_avg = compute_faithfulness(gen_answers, gen_contexts)
print(f"  Faithfulness: {faith_avg:.4f}")

# Summary table
print(f"\n{'=' * 60}")
print(f"  {'Metric':<20s} {'Score':>10s}")
print(f"  {'-' * 35}")
print(f"  {'Precision@5':<20s} {ret_metrics['avg_precision_at_5']:>10.4f}")
print(f"  {'Recall@5':<20s} {ret_metrics['avg_recall_at_5']:>10.4f}")
print(f"  {'MRR':<20s} {ret_metrics['mrr']:>10.4f}")
print(f"  {'ROUGE-L':<20s} {rouge_avg:>10.4f}")
print(f"  {'BERTScore F1':<20s} {bert_avg:>10.4f}")
print(f"  {'Faithfulness':<20s} {faith_avg:>10.4f}")
print(f"{'=' * 60}")

---
## 8. Pre-computed Evaluation Results (All Experiments)

Below we load the full evaluation results from `evaluator.py`, which compared all 6 retrieval experiments and 3 generation strategies.

In [ ]:
# 8.1 — Load and display pre-computed evaluation results
eval_results_file = "evaluation_results/evaluation_results.json"

if os.path.exists(eval_results_file):
    with open(eval_results_file, encoding="utf-8") as f:
        eval_results = json.load(f)
    
    # Retrieval comparison
    print("RETRIEVAL COMPARISON (6 experiments)")
    print(f"{'Method':<12s} {'Model':<6s} {'Chunking':<15s} {'P@5':>6s} {'R@5':>6s} {'MRR':>6s}")
    print("-" * 60)
    for r in eval_results.get("retrieval_evaluation", []):
        print(f"{r['method']:<12s} {r['model']:<6s} {r['chunking']:<15s} "
              f"{r['precision_at_5']:>6.4f} {r['recall_at_5']:>6.4f} {r['mrr']:>6.4f}")
    
    print()
    
    # Generation comparison
    print("GENERATION COMPARISON (3 strategies)")
    print(f"{'Strategy':<14s} {'ROUGE-L':>8s} {'BERTScore':>10s} {'Faithful':>10s} {'Time/q':>8s}")
    print("-" * 55)
    for r in eval_results.get("generation_evaluation", []):
        print(f"{r['strategy']:<14s} {r['rouge_l']:>8.4f} {r['bert_score_f1']:>10.4f} "
              f"{r['faithfulness']:>10.4f} {r['avg_time_per_query']:>7.1f}s")
else:
    print(f"Evaluation results not found. Run: python evaluator.py")

---
## Summary

| Component | Script | Status |
|---|---|---|
| Background Corpus | `build_corpus.py` | 80 documents from 3 sources |
| Chunking | `chunker.py` | 4 strategies compared |
| Embedding | `embedder.py` | 3 models, FAISS indices built |
| Retrieval & Ranking | `retriever.py` | Vector, BM25, Hybrid (top-5 chunks) |
| Generation | `generator.py` | Qwen2.5-0.5B-Instruct, 3 prompt strategies |
| Evaluation | `evaluator.py` | P@5, R@5, MRR, ROUGE-L, BERTScore, Faithfulness |
| Demo App | `demo_app.py` | Streamlit interactive UI |
| **End-to-end Inference** | **`main.ipynb` (this notebook)** | **`run_inference()` — input JSON to output JSON** |

### Best Configuration
- **Retrieval:** Vector + all-mpnet-base-v2 + section_based (MRR 1.0)
- **Prompt:** Structured (ROUGE-L 0.25)
- **Model:** Qwen/Qwen2.5-0.5B-Instruct (required)